# Fresh-split graph-context ensemble v3

Four project-authored 74.7M-parameter members train on a hash-frozen competition split. Selection is frozen before audit; no public predictions, leaderboard selection, metric hack, or submission path is present.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time

import torch

RUN_ID = "competition-graph-context-fresh-ensemble-v3"
EXPECTED_RUNTIME_MANIFEST_SHA256 = "0c7160767dfd48a6d39271578112c1125cd3f486314d788aa7b48775665ed097"
EXPECTED_ARCHIVE_SHA256 = "efa3b5af80c75b2bc091ecda1e86064660dcfffe197e2a4950a12981248b0c3e"
EXPECTED_SPLIT_SHA256 = "8d53a55217be88efc895ae9bf0bf378a3a4acb6c42437836a342d888cc9296da"
RUNTIME = Path("/kaggle/input/biohub-graph-context-fresh-training-runtime-v3")
WORK = Path("/kaggle/working/graph_context_fresh_v3")
OUTPUT = Path("/kaggle/working/graph_context_fresh_ensemble_v3")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest_path = RUNTIME / "runtime_manifest.json"
if sha256_file(manifest_path) != EXPECTED_RUNTIME_MANIFEST_SHA256:
    raise RuntimeError("runtime manifest changed")
manifest = json.loads(manifest_path.read_text())
for relative, record in manifest["files"].items():
    path = RUNTIME / relative
    if path.stat().st_size != record["bytes"] or sha256_file(path) != record["sha256"]:
        raise RuntimeError(f"runtime file changed: {relative}")
if not (
    manifest["patch_archive_sha256"] == EXPECTED_ARCHIVE_SHA256
    and manifest["fresh_split_sha256"] == EXPECTED_SPLIT_SHA256
    and manifest["submission_command_included"] is False
    and manifest["metric_hack_allowed"] is False
    and torch.cuda.is_available()
    and torch.cuda.device_count() == 2
):
    raise RuntimeError("runtime or two-GPU contract failed")
if WORK.exists() or OUTPUT.exists():
    raise RuntimeError("fresh output paths unexpectedly exist")
WORK.mkdir(parents=True)
OUTPUT.mkdir(parents=True)
shutil.copytree(RUNTIME / "research", WORK / "research")
archive = RUNTIME / "biohub_graph_context_relational_patches_v1.tar.gz"
with tarfile.open(archive, "r:gz") as stream:
    stream.extractall(WORK, filter="data")
DATA = WORK / "biohub_graph_context_relational_patches_v1"
SPLIT = RUNTIME / "graph_context_fresh_split_v3.json"
TRAINER = WORK / "research/temporal_contrastive/train_graph_context_fresh_ensemble_v3.py"
TERMINAL = RUNTIME / "pretraining_terminal.json"
print(json.dumps({"gpu_count": torch.cuda.device_count(), "gpu_names": [torch.cuda.get_device_name(i) for i in range(2)], "runtime_verified": True}, sort_keys=True))


In [ ]:
SEEDS = (1409101, 1509107)
FOLDS = (("target_44b6", 1), ("target_6bba", 2))
started = time.monotonic()

def launch_worker(seed, fold, index, gpu):
    member = f"seed-{seed}-init-{index}"
    log_path = OUTPUT / f"{member}.log"
    command = [
        sys.executable, str(TRAINER), "worker",
        "--data-root", str(DATA), "--split", str(SPLIT),
        "--initial-model", str(RUNTIME / "warm_starts" / fold / "pretrained_model.pt"),
        "--warm-start-terminal", str(TERMINAL),
        "--output-root", str(OUTPUT), "--member", member,
        "--seed", str(seed), "--fold", fold,
    ]
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = log_path.open("w", encoding="utf-8")
    process = subprocess.Popen(command, cwd=WORK, env=environment, stdout=log, stderr=subprocess.STDOUT)
    return member, process, log, log_path

for seed in SEEDS:
    workers = [launch_worker(seed, fold, index, gpu) for gpu, (fold, index) in enumerate(FOLDS)]
    deadline = time.monotonic() + 15_000
    while any(process.poll() is None for _, process, _, _ in workers):
        if time.monotonic() > deadline:
            for _, process, _, _ in workers:
                if process.poll() is None:
                    process.terminate()
            raise TimeoutError(f"worker wave exceeded deadline for seed {seed}")
        time.sleep(20)
    failures = []
    for member, process, log, log_path in workers:
        log.close()
        print(f"{member}: exit={process.returncode}; log={log_path}")
        if process.returncode != 0:
            failures.append(member)
    if failures:
        raise RuntimeError(f"worker failures: {failures}")

aggregate_log = OUTPUT / "aggregate.log"
environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"
command = [sys.executable, str(TRAINER), "aggregate", "--data-root", str(DATA), "--split", str(SPLIT), "--output-root", str(OUTPUT)]
with aggregate_log.open("w", encoding="utf-8") as log:
    result = subprocess.run(command, cwd=WORK, env=environment, stdout=log, stderr=subprocess.STDOUT, timeout=3_000, check=False)
terminal_path = OUTPUT / "graph_context_fresh_ensemble_terminal.json"
if not terminal_path.is_file():
    raise RuntimeError(f"aggregate produced no terminal; exit={result.returncode}")
terminal = json.loads(terminal_path.read_text())
if result.returncode not in (0, 2) or terminal.get("run_id") != RUN_ID:
    raise RuntimeError(f"aggregate integrity failed; exit={result.returncode}")
terminal["notebook_elapsed_seconds"] = time.monotonic() - started
terminal["required_visible_gpu_count"] = 2
terminal["runtime_manifest_sha256"] = EXPECTED_RUNTIME_MANIFEST_SHA256
terminal["submission_created"] = False
terminal["authorized_for_submission"] = False
terminal_path.write_text(json.dumps(terminal, indent=2, sort_keys=True) + "\n")
print(json.dumps(terminal, indent=2, sort_keys=True))
